# Section 2 — Numerical Error and Floating-Point Arithmetic

[← Taylor Series](01_taylor_series.ipynb) ·
[Module 1](README.md) ·
[Next: Bisection →](03_bisection_method.ipynb)

Computers do not store arbitrary real numbers exactly. Numerical analysis
therefore tracks both mathematical approximation error and error introduced by
finite-precision arithmetic.

Before studying the IEEE-754 binary format used by modern computers, we begin
with the **finite-digit arithmetic model** used in Burden & Faires. It is a
simple decimal model that makes rounding error visible by hand.

### Learning goals

- compute absolute and relative error;
- use normalized finite-digit floating-point notation;
- apply `fl(...)` after each arithmetic operation;
- distinguish chopping from rounding;
- see why ordinary algebraic laws can fail in finite arithmetic;
- inspect IEEE-754 binary64 fields;
- find the next representable floating-point number;
- understand unit-in-the-last-place (ULP) spacing;
- recognize loss of significance;
- reorganize a formula to reduce numerical error;
- solve inequalities involving a prescribed relative-error tolerance.


In [1]:
import math
import struct


## Absolute and relative error

If $p$ is exact and $p^\ast$ is an approximation,

$$
\text{absolute error}=|p-p^\ast|,
\qquad
\text{relative error}=\frac{|p-p^\ast|}{|p|}.
$$


In [2]:
exact_value = math.pi
approximation = 3.14

absolute_error = abs(exact_value - approximation)
relative_error = absolute_error / abs(exact_value)

print(f"absolute error = {absolute_error:.10f}")
print(f"relative error = {relative_error:.10e}")


absolute error = 0.0015926536
relative error = 5.0695738290e-04


# Finite-digit arithmetic: the Burden & Faires model

A useful model of a decimal floating-point machine stores only a fixed number
$t$ of significant digits. A normalized nonzero number is written in the form

$$
\pm 0.d_1d_2\cdots d_t\times 10^n,
\qquad d_1\ne0.
$$

For example, with $t=3$,

$$
58.5=0.585\times10^2,
\qquad
0.0123=0.123\times10^{-1}.
$$

We use

$$
\operatorname{fl}(x)
$$

to mean **the machine representation of $x$**. In this model, every arithmetic
operation is immediately followed by chopping or rounding.

Thus a computer addition should be thought of as

$$
x\oplus y=\operatorname{fl}(x+y),
$$

rather than exact real-number addition. Likewise,

$$
x\ominus y=\operatorname{fl}(x-y),\qquad
x\otimes y=\operatorname{fl}(xy),\qquad
x\oslash y=\operatorname{fl}(x/y).
$$

This point is crucial: **round after every step, not only at the end.**


## Chopping versus rounding

Suppose only three significant decimal digits are retained.

With **chopping**, all later digits are simply discarded:

$$
\operatorname{fl}_{\rm chop}(1.23456)=1.23.
$$

With **rounding**, the first discarded digit determines whether the final
retained digit changes:

$$
\operatorname{fl}_{\rm round}(1.23656)=1.24.
$$

The small helper functions below simulate the textbook model. The rounding
routine uses the usual classroom convention of rounding a halfway case away
from zero.


In [3]:
def chop(value, digits):
    """Chop a decimal value to a fixed number of significant digits."""
    if value == 0:
        return 0.0

    exponent = math.floor(math.log10(abs(value)))
    scale = 10 ** (digits - 1 - exponent)
    return math.trunc(value * scale) / scale


def round_sig(value, digits):
    """Round a value to significant digits using textbook rounding."""
    if value == 0:
        return 0.0

    exponent = math.floor(math.log10(abs(value)))
    scale = 10 ** (digits - 1 - exponent)
    scaled = value * scale

    if scaled >= 0:
        rounded = math.floor(scaled + 0.5)
    else:
        rounded = math.ceil(scaled - 0.5)

    return rounded / scale


## Course-review example: round or chop after every operation

Using **three-digit chopping arithmetic**, evaluate

$$
(58.5-0.123)+1.96.
$$

The exact real-number calculation is

$$
58.5-0.123+1.96=60.337.
$$

In three-digit chopping arithmetic,

$$
\begin{aligned}
\operatorname{fl}(58.5-0.123)
&=\operatorname{fl}(58.377)\\
&=58.3,
\end{aligned}
$$

and then

$$
\begin{aligned}
\operatorname{fl}(58.3+1.96)
&=\operatorname{fl}(60.26)\\
&=60.2.
\end{aligned}
$$

So the machine calculation gives $60.2$, not $60.337$.

This is the key habit for hand calculations:

> **Every time an arithmetic operation occurs, apply `fl` before continuing.**


In [4]:
a = 58.5
b = -0.123
c = 1.96

first_step = chop(a + b, 3)
left_to_right = chop(first_step + c, 3)
exact_value = a + b + c

absolute_error = abs(exact_value - left_to_right)
relative_error = absolute_error / abs(exact_value)

print(f"first machine result = {first_step}")
print(f"final machine result = {left_to_right}")
print(f"exact value          = {exact_value}")
print(f"absolute error       = {absolute_error:.6f}")
print(f"relative error       = {relative_error:.6e}")


first machine result = 58.3
final machine result = 60.2
exact value          = 60.337
absolute error       = 0.137000
relative error       = 2.270580e-03


## The same expression can give a better answer when reorganized

Real arithmetic lets us regroup addition freely:

$$
(58.5-0.123)+1.96
=
58.5+(-0.123+1.96).
$$

But in finite arithmetic, regrouping changes where rounding occurs.

First calculate the smaller numbers together:

$$
\operatorname{fl}(-0.123+1.96)
=
\operatorname{fl}(1.837)
=
1.83.
$$

Then

$$
\operatorname{fl}(58.5+1.83)
=
\operatorname{fl}(60.33)
=
60.3.
$$

The result $60.3$ is closer to the exact value $60.337$ than $60.2$ was.

This is one of the central lessons of numerical analysis:

> Algebraically equivalent formulas need not be numerically equivalent.


In [5]:
inner = chop(b + c, 3)
reorganized = chop(a + inner, 3)

left_error = abs(exact_value - left_to_right)
reorganized_error = abs(exact_value - reorganized)

print(f"left-to-right result = {left_to_right}")
print(f"reorganized result   = {reorganized}")
print(f"left-to-right error  = {left_error:.6f}")
print(f"reorganized error    = {reorganized_error:.6f}")


left-to-right result = 60.2
reorganized result   = 60.3
left-to-right error  = 0.137000
reorganized error    = 0.037000


# Algebraic laws can fail on a floating-point machine

The real numbers satisfy familiar algebraic identities. A finite set of
machine numbers with rounding after every operation generally does not.

This is not because the algebra is wrong. It is because

$$
\operatorname{fl}(x)
$$

throws away information.

## Example: the distributive law fails

In exact arithmetic,

$$
x(y+z)=xy+xz.
$$

Use **three-digit rounding arithmetic** with

$$
x=0.12,\qquad y=0.50,\qquad z=99.9.
$$

### Left side

First,

$$
\operatorname{fl}(y+z)
=
\operatorname{fl}(100.4)
=
100.
$$

Then,

$$
\operatorname{fl}\left(
x\,\operatorname{fl}(y+z)
\right)
=
\operatorname{fl}(0.12\cdot100)
=
12.0.
$$

### Right side

The products are

$$
\operatorname{fl}(xy)
=
\operatorname{fl}(0.060)
=
0.060,
$$

and

$$
\operatorname{fl}(xz)
=
\operatorname{fl}(11.988)
=
12.0.
$$

Therefore,

$$
\operatorname{fl}\left(
\operatorname{fl}(xy)+\operatorname{fl}(xz)
\right)
=
\operatorname{fl}(12.06)
=
12.1.
$$

Hence, in this three-digit machine,

$$
\boxed{
x\otimes(y\oplus z)\ne
(x\otimes y)\oplus(x\otimes z)
}.
$$

The exact value of both real-number expressions is $12.048$, but the two
machine calculations produce different answers.


In [6]:
x_value = 0.12
y_value = 0.50
z_value = 99.9

left_inner = round_sig(y_value + z_value, 3)
left_side = round_sig(x_value * left_inner, 3)

xy_value = round_sig(x_value * y_value, 3)
xz_value = round_sig(x_value * z_value, 3)
right_side = round_sig(xy_value + xz_value, 3)

print(f"left side  = {left_side}")
print(f"right side = {right_side}")
print(f"exact      = {x_value * (y_value + z_value)}")


left side  = 12.0
right side = 12.1
exact      = 12.048


## Example: associativity of addition can fail too

Exact real arithmetic satisfies

$$
(x+y)+z=x+(y+z).
$$

Now use three-digit rounding with

$$
x=0.001,\qquad y=1,\qquad z=-1.
$$

On the left,

$$
\operatorname{fl}(0.001+1)
=
\operatorname{fl}(1.001)
=
1.00,
$$

so

$$
\operatorname{fl}(1.00-1)=0.
$$

On the right,

$$
\operatorname{fl}(1-1)=0,
$$

so

$$
\operatorname{fl}(0.001+0)=0.001.
$$

Therefore,

$$
\boxed{
(x\oplus y)\oplus z
\ne
x\oplus(y\oplus z)
}.
$$

The small number $0.001$ was **absorbed** when it was added to $1$ before the
subtraction occurred.


In [7]:
x_value = 0.001
y_value = 1.0
z_value = -1.0

left_side = round_sig(
    round_sig(x_value + y_value, 3) + z_value,
    3,
)
right_side = round_sig(
    x_value + round_sig(y_value + z_value, 3),
    3,
)

print(f"(x + y) + z = {left_side}")
print(f"x + (y + z) = {right_side}")


(x + y) + z = 0.0
x + (y + z) = 0.001


## A simple absorption example

With three significant digits,

$$
\operatorname{fl}(1.00+0.001)
=
\operatorname{fl}(1.001)
=
1.00.
$$

The addition took place, but the machine representation cannot retain the
change. This is why adding many tiny quantities to a much larger running total
can be inaccurate.

Later in numerical computation, ideas such as **pairwise summation** and
**compensated summation** are designed to reduce this effect.


In [8]:
large_value = 1.0
small_value = 0.001

machine_sum = round_sig(large_value + small_value, 3)

print(f"exact sum   = {large_value + small_value}")
print(f"machine sum = {machine_sum}")


exact sum   = 1.001
machine sum = 1.0


# IEEE-754 binary64

The decimal model above is deliberately simple enough to use by hand. Real
computers use binary floating-point. Python's ordinary `float` is normally an
IEEE-754 binary64 number:

- 1 sign bit;
- 11 exponent bits;
- 52 fraction bits.

For a normalized binary64 number,

$$
(-1)^s 2^{c-1023}(1+f).
$$

The philosophy is the same as the decimal model: only finitely many numbers
are representable, so arithmetic must be rounded back to a machine number.


In [9]:
def binary64_fields(value):
    """Return sign, exponent, and fraction fields for a Python float."""
    packed = struct.pack(">d", value)
    integer = struct.unpack(">Q", packed)[0]
    bits = f"{integer:064b}"
    return bits[0], bits[1:12], bits[12:]


value = -27.0
sign, exponent, fraction = binary64_fields(value)

print(f"value    = {value}")
print(f"sign     = {sign}")
print(f"exponent = {exponent}")
print(f"fraction = {fraction}")


value    = -27.0
sign     = 1
exponent = 10000000011
fraction = 1011000000000000000000000000000000000000000000000000


## The next machine number and ULP spacing

Floating-point numbers are discrete. If $x$ is a machine number, the **next
representable number** above it is not $x+\varepsilon$ for an arbitrary tiny
real $\varepsilon$; it is the next binary64 pattern.

Python exposes this directly with `math.nextafter`.

The gap

$$
\operatorname{ulp}(x)
$$

is the spacing between nearby representable numbers at the scale of $x$.
This is the computational version of the Exam 1 question asking for the
smallest machine number greater than a given binary64 number.


In [10]:
x_value = 1.0
next_value = math.nextafter(x_value, math.inf)
spacing = math.ulp(x_value)

print(f"x                = {x_value:.17g}")
print(f"next float       = {next_value:.17g}")
print(f"difference       = {next_value - x_value:.17g}")
print(f"math.ulp(x)      = {spacing:.17g}")


x                = 1
next float       = 1.0000000000000002
difference       = 2.2204460492503131e-16
math.ulp(x)      = 2.2204460492503131e-16


## Catastrophic cancellation

Subtracting nearly equal floating-point numbers can destroy significant digits.

For small $c>0$,

$$
\sqrt{x^2+c}-x
$$

subtracts nearby numbers. Rationalizing gives

$$
\frac{c}{\sqrt{x^2+c}+x}.
$$


In [11]:
x_value = 1.0e8
c_value = 1.0

unstable = math.sqrt(x_value**2 + c_value) - x_value
stable = c_value / (math.sqrt(x_value**2 + c_value) + x_value)

print(f"direct formula       = {unstable:.17g}")
print(f"rationalized formula = {stable:.17g}")


direct formula       = 0
rationalized formula = 5.0000000000000001e-09


## Relative-error intervals

A review problem asks for all approximations $p^\ast$ to $p=850$ having relative
error at most $10^{-3}$.

Start from the definition:

$$
\frac{|p-p^\ast|}{|p|}\le10^{-3}.
$$

Thus

$$
|850-p^\ast|\le850(10^{-3})=0.85,
$$

so

$$
\boxed{849.15\le p^\ast\le850.85}.
$$

The important step is to turn a relative-error requirement into an absolute
interval around the true value.


In [12]:
p = 850.0
relative_tolerance = 1.0e-3

radius = abs(p) * relative_tolerance
lower = p - radius
upper = p + radius

print(f"allowable interval = [{lower:.2f}, {upper:.2f}]")


allowable interval = [849.15, 850.85]


### Takeaways

Finite-precision arithmetic behaves like ordinary arithmetic only
approximately.

1. Apply `fl(...)` after **every operation**.
2. Chopping and rounding can produce different results.
3. Changing the order of computation can change the final answer.
4. Associativity and distributivity can fail.
5. Subtraction of nearby numbers can magnify previously introduced error.
6. IEEE-754 is a sophisticated binary implementation of the same basic idea:
   finitely many representable numbers and rounding after arithmetic.

### References

- Richard L. Burden and J. Douglas Faires, *Numerical Analysis*, finite-digit
  arithmetic and floating-point error.
- Leon Q. Brin, *Tea Time Numerical Analysis*.
- MAD 4401 Exam 1 review and course notes.

[← Taylor Series](01_taylor_series.ipynb) ·
[Next: Bisection →](03_bisection_method.ipynb)
